# BIOT 6900 · Module 2 — Multi-Omics Target Identification & Validation
### Week 2 · Computational Lab 2 · Starter Notebook

**What this notebook does.** You'll integrate three data layers — transcriptomics (RNA), proteomics (protein), and genomics — to nominate and *rank* disease targets, then hand that ranked list to Weeks 3–4.

**How today is built — I do → we do → you do:**
- **Part 1 (worked, already complete):** CPTAC **breast cancer** on *synthetic* data. The tumors are measured at all three layers, so the data is *sample-matched* — you integrate **at the sample level**. Your instructor runs these cells; follow along.
- **Part 2 (together, in class):** find and load *real* CPTAC data, then re-run Part 1 on real numbers. Not graded.
- **Part 3 (your assignment, `# TODO` cells):** **Alzheimer's disease**. The cohorts are *not* matched across layers, so you integrate **at the gene level**. You transfer the Part 1 method to messier, more realistic data.

> **The one idea to carry through:** *how* you integrate is dictated by *what your data has matched.* Matched → correlate within samples. Unmatched → compare gene-level summaries. Same goal, different machinery.

**Data.** Part 1 looks for the synthetic CPTAC files in `data/` and, if they're absent, falls back to clearly-labelled demo data so it always runs. Part 3 requires the three Alzheimer's matrices posted on **Canvas** — place them in `data/` (see the Lab Guide).

*Continuity from Module 1:* you'll see **TP53 / p53** (`P04637`, PDB `1TUP`) surface in the cancer half and **APOE** (`rs7412`) in the Alzheimer's half.

**Before you submit:** `Kernel → Restart & Run All` so the whole notebook executes top to bottom.

In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats

RNG = np.random.default_rng(6900)  # fixed seed so results are reproducible

# Module 1 continuity anchors
P53_UNIPROT, P53_PDB = "P04637", "1TUP"   # p53  -> cancer half
APOE_VARIANT = "rs7412"                    # APOE -> Alzheimer's half

# Default scoring weights for this course: equal across the three layers.
# You MAY change these in Part 3 — but if you do, you must justify it in your report.
EQUAL_WEIGHTS = {"transcriptomic": 1/3, "proteomic": 1/3, "genomic": 1/3}

## Helper functions you'll use in both parts
Two small functions do the scoring work. `rank_percentile` puts any layer's scores on a common 0–1 scale (by rank, so it's robust to outliers and to layers being on different units). `multi_evidence_score` normalizes each layer and takes the weighted sum.

In [2]:
def rank_percentile(series: pd.Series) -> pd.Series:
    """Normalize any score to [0, 1] by rank. Robust to outliers and scale differences."""
    return series.rank(method="average", pct=True)


def multi_evidence_score(df: pd.DataFrame, cols, weights) -> pd.Series:
    """Weighted sum of rank-percentile-normalized layer scores."""
    normed = pd.DataFrame({c: rank_percentile(df[c]) for c in cols})
    return sum(weights[c] * normed[c] for c in cols)

## Data loading (provided — you don't need to edit these)
`load_cptac()` reads the matched breast-cancer matrices for Part 1 (or synthesizes demo data if they're missing). `load_ad()` reads the three Alzheimer's matrices you download from Canvas for Part 2.

In [3]:
CPTAC_GENES = ["TP53", "PIK3CA", "ERBB2", "ESR1", "GATA3", "MYC", "CDH1",
               "MAP3K1", "PTEN", "AKT1", "RB1", "CCND1", "FOXA1", "MKI67",
               "EGFR", "BRCA1", "BRCA2", "KRT5", "VIM", "ACTB"]


def _synth_cptac(n_tumor=60, n_normal=15):
    genes = CPTAC_GENES
    tumor_ids = [f"T{i:02d}" for i in range(n_tumor)]
    normal_ids = [f"N{i:02d}" for i in range(n_normal)]
    coupling = pd.Series(RNG.uniform(0.05, 0.80, len(genes)), index=genes)

    def gene_matrix(sample_ids, shift):
        rna = pd.DataFrame(RNG.normal(0, 1, (len(genes), len(sample_ids))),
                           index=genes, columns=sample_ids).add(shift, axis=0)
        noise = pd.DataFrame(RNG.normal(0, 1, rna.shape), index=genes, columns=sample_ids)
        prot = rna.mul(coupling, axis=0) + noise.mul(1 - coupling + 0.35, axis=0)
        return rna, prot

    shift = pd.Series(0.0, index=genes)
    for g in ["TP53", "ERBB2", "MKI67", "MYC", "PIK3CA"]:
        shift[g] = RNG.uniform(1.2, 2.2)
    rna_t, prot_t = gene_matrix(tumor_ids, shift)
    rna_n, prot_n = gene_matrix(normal_ids, pd.Series(0.0, index=genes))
    rna = pd.concat([rna_t, rna_n], axis=1)
    prot = pd.concat([prot_t, prot_n], axis=1)
    mut_freq = pd.Series(RNG.uniform(0.0, 0.05, len(genes)), index=genes)
    for g in ["TP53", "PIK3CA", "CDH1", "GATA3"]:
        mut_freq[g] = RNG.uniform(0.25, 0.45)
    meta = pd.Series(["tumor"] * n_tumor + ["normal"] * n_normal,
                     index=tumor_ids + normal_ids, name="group")
    return rna, prot, mut_freq, meta


def load_cptac():
    p = {"rna": "data/cptac_brca_rna.tsv", "prot": "data/cptac_brca_protein.tsv",
         "mut": "data/cptac_brca_mutation.tsv"}
    if all(os.path.exists(v) for v in p.values()):
        rna = pd.read_csv(p["rna"], sep="\t", index_col=0)
        prot = pd.read_csv(p["prot"], sep="\t", index_col=0)
        mut = pd.read_csv(p["mut"], sep="\t", index_col=0).iloc[:, 0]
        print("Loaded CPTAC files from data/.")
        return rna, prot, mut, None
    print("!! WARNING: CPTAC files not found -> SYNTHETIC demo data "
          "(illustrative only, not real CPTAC values).")
    return _synth_cptac()


def load_ad():
    p = {"tx": "data/ad_transcriptomics.tsv", "pr": "data/ad_proteomics.tsv",
         "gw": "data/ad_gwas.tsv"}
    missing = [v for v in p.values() if not os.path.exists(v)]
    if missing:
        raise FileNotFoundError(
            "Alzheimer's matrices not found: " + ", ".join(missing) +
            "\nDownload the three files from Canvas and put them in a 'data/' folder "
            "next to this notebook (see the Lab Guide).")
    return (pd.read_csv(p["tx"], sep="\t"),
            pd.read_csv(p["pr"], sep="\t"),
            pd.read_csv(p["gw"], sep="\t"))

---
# Part 1 — Worked example: CPTAC breast cancer *(run and read; nothing to edit)*

**Why integrate at all?** Each layer can produce artifacts the others don't share, so a target supported across genome, transcriptome, and proteome is a stronger bet than one seen in only one layer. Integration is triangulation.

Because CPTAC measures the **same tumors** at every layer, we can line samples up and integrate **at the sample level.**

In [4]:
rna, prot, mut_freq, meta = load_cptac()
print("RNA matrix:    ", rna.shape, "(genes x samples)")
print("Protein matrix:", prot.shape)
print("Mutation freq: ", mut_freq.shape)

Loaded CPTAC files from data/.
RNA matrix:     (23121, 122) (genes x samples)
Protein matrix: (12621, 122)
Mutation freq:  (9448,)


### 1.2 Harmonize identifiers → a common gene key
The real technical wall in multi-omics: gene symbols, Ensembl IDs, and UniProt accessions don't line up for free. Here we intersect on a shared key and check how many genes *survive the join* — always your first reality check.

In [5]:
common = rna.index.intersection(prot.index).intersection(mut_freq.index)
print(f"Genes surviving the 3-way join: {len(common)} "
      f"(RNA {len(rna.index)}, protein {len(prot.index)}, mutation {len(mut_freq.index)})")
rna, prot, mut_freq = rna.loc[common], prot.loc[common], mut_freq.loc[common]
samples = rna.columns.intersection(prot.columns)

Genes surviving the 3-way join: 6306 (RNA 23121, protein 12621, mutation 9448)


### 1.3–1.4 Sample-level RNA–protein correlation
With matched samples we can ask, per gene, *does protein track RNA across patients?* The answer is usually **partial** — correlation is modest and varies by gene. A gene where protein does **not** track RNA isn't broken data; it's a signal of post-transcriptional regulation (translational buffering, protein turnover).

In [6]:
corr = pd.Series(
    {g: stats.spearmanr(rna.loc[g, samples], prot.loc[g, samples]).statistic
     for g in common}, name="rna_prot_corr")
print(f"RNA-protein correlation: median={corr.median():.2f}, "
      f"range=[{corr.min():.2f}, {corr.max():.2f}]   <- note: NOT ~1.0")
print(f"  most coupled: {corr.idxmax()} ({corr.max():.2f});  "
      f"most buffered: {corr.idxmin()} ({corr.min():.2f})")

RNA-protein correlation: median=0.48, range=[-0.23, 0.92]   <- note: NOT ~1.0
  most coupled: VWA5A (0.92);  most buffered: ARPC1A (-0.23)


**Read this result.** The median correlation is well below 1 — most genes' protein levels only partly follow their RNA. That is the empirical reason you can't treat RNA as a stand-in for protein, and why integrating both layers adds information.

### 1.5 Per-layer differential signal
Each layer independently nominates candidates. Here the signal is the tumor-vs-normal effect size at RNA and protein, plus per-gene mutation frequency for the genomic layer.

In [7]:
if meta is not None:
    tcols = meta.index[meta == "tumor"]
    ncols = meta.index[meta == "normal"]
    rna_eff = (rna[tcols].mean(axis=1) - rna[ncols].mean(axis=1)).abs()
    prot_eff = (prot[tcols].mean(axis=1) - prot[ncols].mean(axis=1)).abs()
else:
    rna_eff = rna[samples].mean(axis=1).abs()
    prot_eff = prot[samples].mean(axis=1).abs()

scored = pd.DataFrame({"transcriptomic": rna_eff, "proteomic": prot_eff,
                       "genomic": mut_freq, "rna_prot_corr": corr})
scored.round(3).head()

,transcriptomic,proteomic,genomic,rna_prot_corr
A2M,0.060,0.494,0.033,0.349
A2ML1,0.954,3.147,0.016,NaN
AADACL2,0.499,0.396,0.008,NaN
AAED1,0.070,0.434,0.016,NaN
AAGAB,0.026,0.211,0.008,0.666


### 1.6 Multi-evidence score → ranked targets
Normalize each layer to a common scale and take the **equal-weighted** sum. This turns three noisy layers into one ranked list — the artifact everything downstream consumes.

In [8]:
scored["score"] = multi_evidence_score(
    scored, ["transcriptomic", "proteomic", "genomic"], EQUAL_WEIGHTS)
ranked_cptac = scored.sort_values("score", ascending=False)
print("Top CPTAC targets (worked example):")
print(ranked_cptac.head(6).round(3).to_string())
# Continuity check: TP53 / p53 (Module 1's P04637 / 1TUP) is a top breast-cancer hit.
print("\nTP53 rank:", list(ranked_cptac.index).index("TP53") + 1)

Top CPTAC targets (worked example):
         transcriptomic  proteomic  genomic  rna_prot_corr  score
SI                0.977      5.200    0.049            NaN  0.988
VWDE              1.263      1.424    0.049            NaN  0.979
MUC5B             0.566      3.441    0.082            NaN  0.978
SPHKAP            0.573      3.321    0.041            NaN  0.970
CEACAM5           0.813      3.315    0.033            NaN  0.970
RIMS2             1.240      1.676    0.033            NaN  0.968

TP53 rank: 108


### Part 1 recap
You integrated **at the sample level** because the data was **matched** — you could correlate RNA and protein within the same tumors. That was synthetic data, so the method is easy to see. Next we run it on real data, then you transfer it to unmatched data yourself.

---
# Part 2 — Find and load real CPTAC data *(together, in class)*

Now swap the synthetic files for the real thing. CPTAC breast data is open access (no data-use agreement) — we'll get it together in class. **Not graded.**

1. Go to **LinkedOmics** (`linkedomics.org`) → **CPTAC Breast Cancer (BRCA)**.
2. Download the gene-level **RNAseq** and **Proteome** matrices (gene × sample).
3. Save them into `data/` with genes as the row index and these exact names — transpose with `.T` first if samples are in rows:
   - `data/cptac_brca_rna.tsv`
   - `data/cptac_brca_protein.tsv`
   - `data/cptac_brca_mutation.tsv`  (one row per gene: mutation frequency or CNV magnitude)
4. `Kernel → Restart & Run All` and watch **Part 1** again — it now loads your real files instead of the demo. Compare: is the RNA–protein correlation still modest? Does TP53 still surface near the top?

*Wrinkle:* the real download has no tumor/normal labels, so ranking falls back to overall abundance rather than a tumor-vs-normal effect. That's fine for seeing the pipeline run on real data.

---
# Part 3 — Your assignment: Alzheimer's disease *(complete the `# TODO` cells)*

The three matrices you downloaded from Canvas are **gene-level summaries** from different cohorts — there are no shared samples to correlate, so you integrate **at the gene level.** You'll transfer the Part 1 workflow: harmonize → concordance → score → rank → export.

Your ranked target list (the CSV you export in 3.5) is what **Week 3** builds on, so it has to be clean. *Continuity:* expect **APOE** (`rs7412`) among your top hits.

In [9]:
import mygene
print("mygene ready")

mygene ready


In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

def load_luad():

    RAW = Path("data/luad_raw")

    # Load the five CPTAC-LUAD raw files
    rna_tumor = pd.read_csv(
        next(RAW.glob("*RNAseq*Tumor.txt")), sep="\t"
    )
    rna_normal = pd.read_csv(
        next(RAW.glob("*RNAseq*Normal.txt")), sep="\t"
    )

    prot_tumor = pd.read_csv(
        next(RAW.glob("*proteomics*Tumor.txt")), sep="\t"
    )
    prot_normal = pd.read_csv(
        next(RAW.glob("*proteomics*Normal.txt")), sep="\t"
    )

    mutation = pd.read_csv(
        next(RAW.glob("*somatic_mutation*gene_level*")), sep="\t"
    )

    # Remove Ensembl version suffixes
    for d in [rna_tumor, rna_normal, prot_tumor, prot_normal, mutation]:
        d["ensembl"] = d["idx"].str.split(".").str[0]

    # ---------------- RNA ----------------
    rna_pairs = sorted(
        (set(rna_tumor.columns) & set(rna_normal.columns))
        - {"idx", "ensembl"}
    )

    rt = (
        rna_tumor[["ensembl"] + rna_pairs]
        .groupby("ensembl")
        .mean()
    )

    rn = (
        rna_normal[["ensembl"] + rna_pairs]
        .groupby("ensembl")
        .mean()
    )

    common_rna = rt.index.intersection(rn.index)
    rt = rt.loc[common_rna]
    rn = rn.loc[common_rna]

    tx_raw = pd.DataFrame({
        "ensembl": common_rna,
        "log2fc": (rt - rn).mean(axis=1).values,
        "pval": stats.ttest_rel(
            rt.to_numpy(),
            rn.to_numpy(),
            axis=1,
            nan_policy="omit"
        ).pvalue
    })

    # ---------------- Protein ----------------
    prot_pairs = sorted(
        (set(prot_tumor.columns) & set(prot_normal.columns))
        - {"idx", "ensembl"}
    )

    pt = (
        prot_tumor[["ensembl"] + prot_pairs]
        .groupby("ensembl")
        .mean()
    )

    pn = (
        prot_normal[["ensembl"] + prot_pairs]
        .groupby("ensembl")
        .mean()
    )

    common_prot = pt.index.intersection(pn.index)
    pt = pt.loc[common_prot]
    pn = pn.loc[common_prot]

    pr_raw = pd.DataFrame({
        "ensembl": common_prot,
        "log2fc": (pt - pn).mean(axis=1, skipna=True).values,
        "pval": stats.ttest_rel(
            pt.to_numpy(),
            pn.to_numpy(),
            axis=1,
            nan_policy="omit"
        ).pvalue
    })

    # ---------------- Genomics ----------------
    mut_samples = [
        c for c in mutation.columns
        if c not in ["idx", "ensembl"]
    ]

    mut_matrix = (
        mutation[["ensembl"] + mut_samples]
        .groupby("ensembl")
        .mean()
    )

    gw_raw = pd.DataFrame({
        "ensembl": mut_matrix.index,
        "mutation_freq": mut_matrix.mean(axis=1).values
    })

    # ---------------- Ensembl -> gene symbol ----------------
    common_ids = sorted(
        set(tx_raw["ensembl"])
        & set(pr_raw["ensembl"])
        & set(gw_raw["ensembl"])
    )

    mg = mygene.MyGeneInfo()

    results = mg.querymany(
        common_ids,
        scopes="ensembl.gene",
        fields="symbol",
        species="human"
    )

    mapping = pd.DataFrame([
        {
            "ensembl": x.get("query"),
            "gene": x.get("symbol")
        }
        for x in results
        if x.get("symbol") is not None
    ]).drop_duplicates("ensembl")

    tx = (
        tx_raw.merge(mapping, on="ensembl")
        [["gene", "log2fc", "pval"]]
    )

    pr = (
        pr_raw.merge(mapping, on="ensembl")
        [["gene", "log2fc", "pval"]]
    )

    gw = (
        gw_raw.merge(mapping, on="ensembl")
        [["gene", "mutation_freq"]]
    )

    return tx, pr, gw

In [11]:
import mygene
print("mygene ready")

mygene ready


### 3.1 — TODO: load and inspect the three matrices
Call `load_ad()` and look at each table's columns and shape before you touch them.

In [12]:
# TODO 3.1 — load the three LUAD matrices and inspect them.
tx, pr, gw = load_luad()

print("Transcriptomics:", tx.shape, tx.columns.tolist())
print("Proteomics:", pr.shape, pr.columns.tolist())
print("Genomics:", gw.shape, gw.columns.tolist())

display(tx.head())
display(pr.head())
display(gw.head())

C:\Users\kanap\AppData\Local\Temp\ipykernel_25312\759653707.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["ensembl"] = d["idx"].str.split(".").str[0]
1 input query terms found no hit:	['ENSG00000130723']


Transcriptomics: (6758, 3) ['gene', 'log2fc', 'pval']
Proteomics: (6758, 3) ['gene', 'log2fc', 'pval']
Genomics: (6758, 2) ['gene', 'mutation_freq']


,gene,log2fc,pval
0,DPM1,0.229971,2.443424e-06
1,SCYL3,0.408261,9.268772e-16
2,FIRRM,1.063954,9.084881e-31
3,FGR,-1.884782,3.154438e-46
4,CFH,-0.181050,1.840694e-02


,gene,log2fc,pval
0,DPM1,0.186768,8.045239e-09
1,SCYL3,0.256987,2.858806e-10
2,FIRRM,0.102737,3.012266e-01
3,FGR,-0.585544,3.892596e-24
4,CFH,-0.683658,3.402547e-26


,gene,mutation_freq
0,DPM1,0.009259
1,SCYL3,0.009259
2,FIRRM,0.009259
3,FGR,0.018519
4,CFH,0.037037


### 3.2 — TODO: harmonize on the gene symbol → one joined table
Rename the effect/`pval` columns so RNA and protein don't collide, then merge all three on `gene`. Report how many genes survive the join (your first reality check).

In [13]:
# TODO 3.2 — build a single joined table `df`.
# Hint: rename before merging so columns don't clash.

tx = tx.rename(columns={"log2fc": "rna_lfc", "pval": "rna_p"})
pr = pr.rename(columns={"log2fc": "prot_lfc", "pval": "prot_p"})

df = tx.merge(pr, on="gene").merge(gw, on="gene")

print(f"Genes surviving the 3-way join: {len(df)}")

Genes surviving the 3-way join: 6758


### 3.3 — TODO: sign-agreement concordance
You can't correlate across samples here (nothing is matched), so concordance becomes **direction agreement**: does the gene move the *same way* at RNA and protein? Add a boolean `concordant` column. Watch for genes that are strong at RNA but flat/opposite at protein — those are the interesting discordant ones.

In [14]:
# TODO 3.3 — add a boolean `concordant` column: do RNA and protein point the same direction?
# Hint: compare np.sign(df["rna_lfc"]) with np.sign(df["prot_lfc"]).
df["concordant"] = np.sign(df["rna_lfc"]) == np.sign(df["prot_lfc"])
print(f"Sign-concordant genes: {df['concordant'].sum()} / {len(df)}")

Sign-concordant genes: 4918 / 6758


### 3.4 — TODO: multi-evidence score
Build a per-layer magnitude for each of the three layers, then score with `multi_evidence_score` and `EQUAL_WEIGHTS`. If you change the weights, justify it in your report.

In [15]:
df["transcriptomic"] = df["rna_lfc"].abs()
df["proteomic"]      = df["prot_lfc"].abs()
df["genomic"]        = df["mutation_freq"]
df["score"] = multi_evidence_score(df, ["transcriptomic", "proteomic", "genomic"], EQUAL_WEIGHTS)
print(f"Scored {len(df)} genes; score range [{df['score'].min():.3f}, {df['score'].max():.3f}]")

Scored 6758 genes; score range [0.084, 0.992]


### 3.5 — TODO: rank, inspect, and export
Sort by score, take the top ~15, and **export the ranked table to `targets_ad.csv`** — this file is the hand-off to Week 3. Check whether known AD genes (APOE, TREM2, BIN1, CLU, PICALM …) are recovered, and look at any `concordant == False` genes in your top hits.

In [16]:
# TODO 3.5 — rank, look at the top 15, and export the CSV.
ranked = df.sort_values("score", ascending=False)
top = ranked.head(15)

top.to_csv("targets_luad.csv", index=False)

print(
    top[
        ["gene", "rna_lfc", "prot_lfc",
         "mutation_freq", "concordant", "score"]
    ].round(3).to_string(index=False)
)

    gene  rna_lfc  prot_lfc  mutation_freq  concordant  score
   ACTN2   -3.447    -1.731          0.120        True  0.992
  ZNF536   -2.716     2.712          0.120       False  0.988
  COL6A6   -3.042    -1.885          0.083        True  0.986
     TNR   -3.763    -1.266          0.111        True  0.985
   SCN7A   -2.936    -1.407          0.093        True  0.982
   ABCA8   -3.257    -1.581          0.056        True  0.976
ADAMTS16    2.558     1.234          0.102        True  0.973
    TNXB   -2.952    -1.456          0.056        True  0.972
    NOS1   -3.691    -1.069          0.074        True  0.972
    MYH2   -2.726    -1.611          0.056        True  0.971
   MASP1   -3.403    -1.135          0.065        True  0.970
  ERICH3   -3.340    -0.989          0.093        True  0.969
 COL11A1    6.157     0.871          0.204        True  0.968
  SEMA5A   -2.431    -1.096          0.120        True  0.967
   GDF10   -3.710    -1.354          0.046        True  0.967


### 3.6 — Interpretation (write-up, goes in your report)
Answer these in your 3–4 page report — this is the 40% interpretation payload:

1. **Weighting.** You used equal weights. Argue for keeping them equal *or* for up-weighting a layer (e.g. GWAS as germline/causal-leaning vs. transcriptomics as possibly downstream). There's no single right answer — only reasoned vs. unreasoned.
2. **Top targets.** Which known AD genes did you recover (APOE, TREM2, BIN1, CLU, PICALM …)? Any non-obvious hit worth a second look?
3. **Read a discordant gene.** Pick a `concordant == False` gene in your top hits (strong RNA/GWAS, flat protein). What biology could explain RNA and protein disagreeing?
4. **Limitation.** This was **gene-level, cross-cohort** integration — unmatched — so you *cannot* make per-patient claims. Contrast this with the matched CPTAC case from Part 1.

---
### Submit (Part 3 only)
1. `Kernel → Restart & Run All` — confirm the whole notebook runs top to bottom.
2. Commit **this notebook**, **`targets_ad.csv`**, and a short **README** (your name + anything that didn't work) to your `biot6900` repo.
3. Push, confirm the files appear on github.com, then **post your repo link on Canvas.**

Grading follows the course 60 / 40 split — 60 execution, 40 interpretation & communication. Partial credit for a correct approach even with minor technical errors: if a step wouldn't run, say what you were trying to do and what happened.